In [ ]:
# New Conquer Offline — extract the complete XAPK into the Godot project (single cell)
# Downloads the existing Git LFS source, verifies it, extracts every archive entry,
# creates SHA-256 manifests, and commits the extracted tree back to main using Git LFS.

import os, sys, json, csv, hashlib, zipfile, shutil, subprocess, pathlib, getpass, textwrap
from IPython.display import display, Markdown

OWNER, REPO, BRANCH = "ZzwahwahzZ", "New-Conquer-Offline", "main"
EXPECTED_SIZE = 461098460
EXPECTED_SHA256 = "7948a626a92f4e5091f65f7cc47a1c4caf1991cd201dcb984eb4606fb261fa83"
REPO_DIR = pathlib.Path("/content/New-Conquer-Offline")
XAPK_REL = pathlib.Path("source_game/original/conquer-source.xapk")
EXTRACT_REL = pathlib.Path("source_game/extracted")
REPORT_REL = pathlib.Path("source_game/extraction_reports")

display(Markdown("## استخراج نسخة كاملة من XAPK إلى مشروع Godot"))
token = getpass.getpass("GitHub token (لن يظهر أثناء الكتابة): ").strip()
if not token:
    raise ValueError("لم يتم إدخال GitHub token.")

def run(cmd, cwd=None, env=None):
    print("+", " ".join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout:
        print(p.stdout[-12000:])
    if p.returncode:
        raise RuntimeError(f"فشل الأمر برمز {p.returncode}: {cmd[0]}")
    return p.stdout

# Use askpass so the token is not embedded in the repository URL or printed in commands.
askpass = pathlib.Path("/content/gh-askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GH_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "GIT_LFS_SKIP_SMUDGE": "1"})

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, f"https://github.com/{OWNER}/{REPO}.git", str(REPO_DIR)], env=env)
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"المسار موجود لكنه ليس مستودع Git: {REPO_DIR}")
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, env=env)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR, env=env)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, env=env)

run(["git", "lfs", "install", "--local"], cwd=REPO_DIR, env=env)
run(["git", "lfs", "pull", "origin", BRANCH], cwd=REPO_DIR, env=env)

xapk = REPO_DIR / XAPK_REL
if not xapk.is_file():
    raise FileNotFoundError(f"ملف المصدر غير موجود بعد Git LFS pull: {xapk}")
actual_size = xapk.stat().st_size
h = hashlib.sha256()
with xapk.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)
actual_sha = h.hexdigest()
print(f"Source size: {actual_size:,} bytes")
print("Source SHA256:", actual_sha)
if actual_size != EXPECTED_SIZE or actual_sha != EXPECTED_SHA256:
    raise RuntimeError("فشل التحقق: الحجم أو SHA-256 لا يطابق النسخة الأصلية المعروفة.")

extract_dir = REPO_DIR / EXTRACT_REL
if extract_dir.exists() and any(extract_dir.iterdir()):
    raise FileExistsError(f"مجلد الاستخراج يحتوي ملفات بالفعل؛ أوقفت العملية حتى لا أستبدل أي شيء: {extract_dir}")
extract_dir.mkdir(parents=True, exist_ok=True)
reports_dir = REPO_DIR / REPORT_REL
reports_dir.mkdir(parents=True, exist_ok=True)

records = []
total_uncompressed = 0
with zipfile.ZipFile(xapk, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"الأرشيف تالف عند اختبار الملف: {bad}")
    infos = [i for i in zf.infolist() if not i.is_dir()]
    names = set()
    for info in infos:
        rel = pathlib.PurePosixPath(info.filename)
        if rel.is_absolute() or ".." in rel.parts or not rel.parts:
            raise RuntimeError(f"مسار غير آمن داخل الأرشيف، أوقفت الاستخراج: {info.filename}")
        if info.filename in names:
            raise RuntimeError(f"اسم مكرر داخل الأرشيف ولا يمكن ضمان الحفاظ على النسختين: {info.filename}")
        names.add(info.filename)
        total_uncompressed += info.file_size
    print(f"Archive entries: {len(infos):,} files")
    print(f"Uncompressed total: {total_uncompressed / (1024**3):.2f} GiB")
    for index, info in enumerate(infos, 1):
        target = extract_dir.joinpath(*pathlib.PurePosixPath(info.filename).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        digest = hashlib.sha256()
        written = 0
        with zf.open(info, "r") as src, target.open("wb") as dst:
            while True:
                chunk = src.read(8 * 1024 * 1024)
                if not chunk:
                    break
                dst.write(chunk)
                digest.update(chunk)
                written += len(chunk)
        if written != info.file_size:
            raise RuntimeError(f"الحجم المستخرج لا يطابق الأرشيف: {info.filename}")
        records.append({
            "path": info.filename,
            "size_bytes": written,
            "sha256": digest.hexdigest(),
            "extension": pathlib.PurePosixPath(info.filename).suffix.lower(),
            "zip_compressed_bytes": info.compress_size
        })
        if index % 50 == 0 or index == len(infos):
            print(f"Extracted {index}/{len(infos)}")

# Verify every archive file was extracted byte-for-byte; the outer XAPK remains separate.
verified = 0
for record in records:
    extracted_file = extract_dir.joinpath(*pathlib.PurePosixPath(record['path']).parts)
    if not extracted_file.is_file() or extracted_file.stat().st_size != record['size_bytes']:
        raise RuntimeError(f"ملف مفقود أو حجمه غير مطابق: {record['path']}")
    check = hashlib.sha256()
    with extracted_file.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            check.update(chunk)
    if check.hexdigest() != record['sha256']:
        raise RuntimeError(f"فشل تحقق SHA-256: {record['path']}")
    verified += 1
if verified != len(infos):
    raise RuntimeError('عدد الملفات المتحقق منها لا يساوي عدد ملفات الأرشيف.')
apk_records = [r for r in records if r['extension'] == '.apk']
if not apk_records:
    raise RuntimeError('لم يتم العثور على أي APK داخل XAPK؛ أوقفت العملية.')
print(f'Verified extracted files: {verified}/{len(infos)}')
print('APK files found:')
for item in apk_records:
    print(' -', item['path'], f"({item['size_bytes']:,} bytes)")
if (extract_dir / xapk.name).exists():
    raise RuntimeError('ملف XAPK الخارجي ظهر داخل مجلد المحتويات المستخرجة.')
print('Outer XAPK remains separate at:', xapk)

# Save detailed inventories outside the extracted tree so the source files remain untouched.
manifest_json = reports_dir / "manifest.json"
manifest_csv = reports_dir / "manifest.csv"
manifest_json.write_text(json.dumps({
    "source_file": str(XAPK_REL),
    "source_size_bytes": actual_size,
    "source_sha256": actual_sha,
    "archive_file_count": len(records),
    "total_uncompressed_bytes": total_uncompressed,
    "files": records
}, ensure_ascii=False, indent=2))
with manifest_csv.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["path", "size_bytes", "sha256", "extension", "zip_compressed_bytes"])
    writer.writeheader()
    writer.writerows(records)

# Basic type inventory for a quick view of what the XAPK contains.
from collections import Counter
ext_counts = Counter(r["extension"] or "[no extension]" for r in records)
summary = [
    "# XAPK extraction inventory",
    "",
    f"- Source: `{XAPK_REL}`",
    f"- Source SHA-256: `{actual_sha}`",
    f"- Extracted files: **{len(records)}**",
    f"- Total extracted size: **{total_uncompressed:,} bytes**",
    "",
    "## File types",
]
summary += [f"- {ext}: {count}" for ext, count in sorted(ext_counts.items())]
summary += ["", "Every non-directory archive entry was extracted with its relative path preserved.", "See `../extraction_reports/manifest.json` and `manifest.csv` for per-file sizes and SHA-256 hashes."]
(REPO_DIR / EXTRACT_REL / "README.txt").write_text("\n".join(summary), encoding="utf-8")

# Git LFS stores the extracted binary tree as LFS objects, not huge ordinary Git blobs.
too_large = [r for r in records if r["size_bytes"] >= 2 * 1024**3]
if too_large:
    raise RuntimeError("توجد ملفات فردية بحجم 2 GiB أو أكثر لا يقبلها Git LFS على GitHub؛ لم أبدأ الرفع. راجع manifest.json.")
run(["git", "lfs", "track", "source_game/extracted/**"], cwd=REPO_DIR, env=env)
run(["git", "add", ".gitattributes", str(EXTRACT_REL), str(REPORT_REL)], cwd=REPO_DIR, env=env)
status = run(["git", "status", "--short"], cwd=REPO_DIR, env=env)
if not status.strip():
    raise RuntimeError("لا توجد تغييرات جديدة لرفعها؛ راجع هل تم استخراج المحتويات مسبقًا.")
run(["git", "config", "user.name", "New Conquer Offline Asset Import"], cwd=REPO_DIR, env=env)
run(["git", "config", "user.email", "actions@users.noreply.github.com"], cwd=REPO_DIR, env=env)
run(["git", "commit", "-m", "Import complete extracted XAPK contents into Godot project"], cwd=REPO_DIR, env=env)
run(["git", "push", "origin", BRANCH], cwd=REPO_DIR, env=env)

# Confirm the pushed commit and tracked LFS inventory.
commit = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, env=env).strip()
lfs_list = run(["git", "lfs", "ls-files", str(EXTRACT_REL)], cwd=REPO_DIR, env=env)
print("\n✅ اكتمل الاستخراج والرفع إلى المستودع.")
print("Extracted files:", len(records))
print("Extracted bytes:", f"{total_uncompressed:,}")
print("APK files:", sum(1 for r in records if r["extension"] == ".apk"))
print("OBB files:", sum(1 for r in records if r["extension"] == ".obb"))
print("Commit:", commit)
print("LFS tracked extracted entries:", len([line for line in lfs_list.splitlines() if line.strip()]))
print("Project folder:", f"https://github.com/{OWNER}/{REPO}/tree/{BRANCH}/source_game/extracted")
print("Manifest:", f"https://github.com/{OWNER}/{REPO}/tree/{BRANCH}/source_game/extraction_reports")
print("Source APK/XAPK remains preserved; no existing project files were deleted or overwritten.")
